# Anthropic이 공개한 에이전트 메모리 4계층과 dreaming의 효과를 쉽게 풀고, 공식 knowledge-wiki 예제로 Claude Managed Agents 메모리를 써 보는 법을 쓴다

위에서부터 차례로 실행해 보세요.

### 준비: API 키 넣기

Colab 왼쪽의 🔑(보안 비밀)에 `ANTHROPIC_API_KEY` 를 자기 값으로 추가하고 노트북 액세스를 켜 주세요. 키는 노트북에 저장되지 않아요.

In [ ]:
import os
from google.colab import userdata

for name in ['ANTHROPIC_API_KEY']:
    os.environ[name] = userdata.get(name)

### 저장소 받기

In [ ]:
%%bash
git clone --depth 1 https://github.com/anthropics/claude-quickstarts.git cq
cd cq/managed-agents/knowledge-wiki

### quickstart 공시 받기

In [ ]:
%%bash
cd cq/managed-agents/knowledge-wiki
export EDGAR_USER_AGENT="your-name your-email"
python3 fetch_data_room.py --tier=quickstart
ls -l data_room/docs
head -c 600 data_room/docs/4_20241016_2586.txt

### 저장소 받기와 설치, 공시 문서 받기

In [ ]:
%%bash
git clone --depth 1 https://github.com/anthropics/claude-quickstarts.git
cd claude-quickstarts/managed-agents/knowledge-wiki
pip install -q -r requirements.txt
export EDGAR_USER_AGENT="your-name your-email"
rm -f data_room/docs/*
python3 fetch_data_room.py --tier=quickstart
ls data_room/docs

### 예산 1달러로 비용 추정과 중단 확인 (ANTHROPIC_API_KEY 필요)

In [ ]:
%%bash
cd claude-quickstarts/managed-agents/knowledge-wiki
python3 - <<'E'
import json
nb=json.load(open('distill_documents_into_knowledge_wiki.ipynb'))
keep=[c for i,c in enumerate(nb['cells']) if i<=16 and not ''.join(c['source']).lstrip().startswith('%%capture')]
for c in keep:
    if c['cell_type']=='code': c['outputs']=[]
nb['cells']=keep
json.dump(nb,open('budget_demo.ipynb','w'))
E
COOKBOOK_BUDGET_USD=1 jupyter nbconvert --to notebook --execute budget_demo.ipynb --output budget_out.ipynb 2>&1 | tail -15

### 예산 10달러로 위키 구축 후 파일 구조 확인 (약 16분, ANTHROPIC_API_KEY 필요)

In [ ]:
%%bash
cd claude-quickstarts/managed-agents/knowledge-wiki
python3 - <<'E'
import json
nb=json.load(open('distill_documents_into_knowledge_wiki.ipynb'))
keep=[c for i,c in enumerate(nb['cells']) if (i<=35 or i in (41,42)) and not ''.join(c['source']).lstrip().startswith('%%capture')]
for c in keep:
    if c['cell_type']=='code': c['outputs']=[]
nb['cells']=keep
json.dump(nb,open('run_qs.ipynb','w'))
E
COOKBOOK_BUDGET_USD=10 COOKBOOK_RECORDS_DIR=/content/rec jupyter nbconvert --to notebook --execute run_qs.ipynb --output run_qs_out.ipynb --ExecutePreprocessor.timeout=3000 2>&1 | tail -5
find /content/rec -path '*wiki/*' -type f | sort | head -70

### 위키 답변의 출처 인용 확인 (노트북 실행 결과에서 질문·답 출력)

In [ ]:
%%bash
cd claude-quickstarts/managed-agents/knowledge-wiki
python3 - <<'E'
import json
nb=json.load(open('run_qs_out.ipynb'))
for c in nb['cells']:
    if c['cell_type']=='code' and ('q_price = ask' in ''.join(c['source']) or 'q_trap = ask' in ''.join(c['source'])):
        for o in c['outputs']:
            if 'data' in o: print(''.join(o['data'].get('text/markdown','')))
E

### EBITDA 충돌이 위키에 남았는지 확인

In [ ]:
%%bash
cd claude-quickstarts/managed-agents/knowledge-wiki
grep -rn "CONFLICT (EBITDA" /content/rec/*/build_the_knowledge_wiki/wiki/entities/squarespace.md
cat /content/rec/*/build_the_knowledge_wiki/wiki/metrics/analyst-note.md